# Animal Detection with YOLOv8 and Roboflow

This notebook demonstrates an end-to-end pipeline for training a YOLOv8 object detection model using a dataset from Roboflow. It includes data preparation, model training, evaluation, and a custom visualization to highlight detected objects by drawing on them directly, as per the new requirements.

## 1. Environment Setup & Dataset Preparation

In [ ]:
# Install required packages
!pip install ultralytics roboflow opencv-python matplotlib pandas scikit-learn pyyaml

# Import necessary libraries
import os
import cv2
import yaml
import torch
import roboflow
import numpy as np
import matplotlib.pyplot as plt
from ultralytics import YOLO
from PIL import Image

print("Setup complete. All libraries are installed and imported.")

### 1.2 Download Dataset from Roboflow

We will use the Roboflow API to download our dataset. You need to replace `"YOUR_ROBOFLOW_API_KEY"` with your actual private API key from Roboflow.

The selected dataset is `animal-detection-c5/1`, which contains 5 classes and over 2,000 images, suitable for our training purposes.

In [ ]:
# Authenticate with Roboflow
rf = roboflow.Roboflow(api_key="YOUR_ROBOFLOW_API_KEY") # Replace with your key

# Download the dataset
project = rf.workspace("roboflow-jvuqo").project("animal-detection-c5")
dataset = project.version(1).download("yolov8")

# Get the path to the data.yaml file
data_yaml_path = os.path.join(dataset.location, "data.yaml")
print(f"Dataset downloaded to: {dataset.location}")
print(f"data.yaml path: {data_yaml_path}")

## 2. Model Configuration & Training

Now, we'll train the YOLOv8 model on our custom dataset. We'll use `yolov8n.pt` as our base model and train it for 100 epochs.

In [ ]:
# Load a pre-trained YOLO model
model = YOLO('yolov8n.pt')

# Train the model
results = model.train(
    data=data_yaml_path,
    epochs=100,
    imgsz=640,
    batch=16,
    device='auto',
    patience=20,
    save_period=10,
    project="animal_detection_training",
    name="yolov8n_100_epochs"
)

print("Training finished.")

## 3. Model Evaluation & Validation

After training, we evaluate the model's performance on the test set to understand its accuracy and identify areas for improvement.

In [ ]:
# Load the best performing model
best_model_path = os.path.join(results.save_dir, 'weights/best.pt')
model = YOLO(best_model_path)

# Run validation
metrics = model.val(split='test')

# Print key metrics
print(f"mAP@50-95: {metrics.box.map}")
print(f"mAP@50: {metrics.box.map50}")
print(f"Precision: {metrics.box.p[0]}")
print(f"Recall: {metrics.box.r[0]}")

### 3.1 Confusion Matrix

The confusion matrix helps visualize the performance of the classification aspect of our model.

In [ ]:
# Display the confusion matrix
confusion_matrix_path = os.path.join(metrics.save_dir, 'confusion_matrix.png')
if os.path.exists(confusion_matrix_path):
    img = Image.open(confusion_matrix_path)
    plt.figure(figsize=(10, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Confusion Matrix')
    plt.show()
else:
    print("Confusion matrix not found.")

## 4. Inference & Visualization

This section focuses on running inference with our trained model and implementing the custom visualization to draw overlays on detected objects.

### 4.1 Custom Visualization: Drawing on Objects

Instead of drawing a bounding box *around* the object, we will draw a semi-transparent overlay *on* the object itself. This provides a clearer visualization of what the model has identified.

In [ ]:
# Get a few test images
test_img_dir = os.path.join(dataset.location, "test/images")
test_images = [os.path.join(test_img_dir, img) for img in os.listdir(test_img_dir)[:5]]

# Define colors for each class
colors = [
    (255, 0, 0),    # Red
    (0, 255, 0),    # Green
    (0, 0, 255),    # Blue
    (255, 255, 0),  # Yellow
    (255, 0, 255),  # Magenta
]

for img_path in test_images:
    # Read the image
    img = cv2.imread(img_path)
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    overlay = img.copy()

    # Run inference
    results = model(img_path)

    for r in results:
        for i, box in enumerate(r.boxes.xyxy):
            x1, y1, x2, y2 = map(int, box)
            class_id = int(r.boxes.cls[i])
            conf = float(r.boxes.conf[i])
            label = f"{model.names[class_id]} {conf:.2f}"
            
            # Get color for the class
            color = colors[class_id % len(colors)]

            # Draw a filled rectangle on the overlay
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
            
            # Add label
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Blend the overlay with the original image
    alpha = 0.4  # Transparency factor
    final_img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)
    final_img_rgb = cv2.cvtColor(final_img, cv2.COLOR_BGR2RGB)

    # Display the image
    plt.figure(figsize=(10, 10))
    plt.imshow(final_img_rgb)
    plt.axis('off')  # Hide the axes
    plt.title(f"Detections for {os.path.basename(img_path)}")
    plt.show()

## 5. Documentation & Export

Finally, we'll export the model and create the necessary files for deployment and reproducibility.

### 5.1 Export to ONNX

Exporting the model to ONNX format makes it portable and allows it to be used in various deployment environments.

In [ ]:
# Export the model to ONNX format
onnx_path = model.export(format='onnx')
print(f"Model exported to {onnx_path}")

### 5.2 Create `requirements.txt`

In [ ]:
requirements = """
ultralytics
roboflow
opencv-python
matplotlib
pandas
scikit-learn
pyyaml
torch
"""

with open("requirements.txt", "w") as f:
    f.write(requirements)

print("requirements.txt created.")

### 5.3 Create `inference.py`

This standalone script can be used to run inference from the command line.

In [ ]:
inference_script = """
import cv2
import os
import argparse
from ultralytics import YOLO
import numpy as np

def run_inference(image_path, model_path='best.pt'):
    # Load the model
    model = YOLO(model_path)

    # Read the image
    img = cv2.imread(image_path)
    if img is None:
        print(f"Error: Could not read image at {image_path}")
        return

    overlay = img.copy()

    # Run inference
    results = model(image_path)
    
    # Define colors for each class
    colors = [
        (255, 0, 0), (0, 255, 0), (0, 0, 255),
        (255, 255, 0), (255, 0, 255)
    ]

    for r in results:
        for i, box in enumerate(r.boxes.xyxy):
            x1, y1, x2, y2 = map(int, box)
            class_id = int(r.boxes.cls[i])
            conf = float(r.boxes.conf[i])
            label = f"{model.names[class_id]} {conf:.2f}"
            
            color = colors[class_id % len(colors)]

            # Draw a filled rectangle on the overlay
            cv2.rectangle(overlay, (x1, y1), (x2, y2), color, -1)
            
            # Add label
            cv2.putText(img, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

    # Blend the overlay with the original image
    alpha = 0.4
    final_img = cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0)

    # Save the output
    output_dir = 'output'
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, os.path.basename(image_path))
    cv2.imwrite(output_path, final_img)
    print(f"Inference result saved to {output_path}")

if __name__ == '__main__':
    parser = argparse.ArgumentParser(description='Run YOLO inference on an image.')
    parser.add_argument('image_path', type=str, help='Path to the input image.')
    parser.add_argument('--model_path', type=str, default='best.pt', help='Path to the trained model weights.')
    args = parser.parse_args()
    
    run_inference(args.image_path, args.model_path)
"""

with open("inference.py", "w") as f:
    f.write(inference_script)

print("inference.py created.")